In [ ]:
# Set Spark config to access ADLS Gen2
spark.conf.set(
    "fs.azure.account.key.<adls-storage-account>.dfs.core.windows.net",
    "<access-key>"
)

In [ ]:
# Path to the Parquet files on ADLS Gen2 using abfss protocol
parquet_path = "abfss://<adls-container>@<adls-storage>.dfs.core.windows.net/silver"

In [ ]:
# Load the dataset from Parquet stored in ADLS
gold_df = spark.read.option("header", True).parquet(parquet_path)

In [ ]:
spark.conf.set("spark.gql.catalog.iceberg_catalog", "org.apache.iceberg.spark.Sparkcatalog")
spark.conf.set("spark.sql.catalog.iceberg_catalog.type", "hadoop")
spark.conf.set("spark.sql.catalog.iceberg_catalog.warehouse", "abfss://<adls-container>@<adls-storage>.dfs.core.windows.net/iceberg_warehouse/"
)

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS iceberg_catalog.iceberg_db")


In [ ]:
# Drop existing table if needed
spark.sql("DROP TABLE IF EXISTS iceberg_catalog.iceberg_db.parking_iceberg")


In [ ]:
gold_df.writeTo("iceberg_catalog.iceberg_db.parking_iceberg").createOrReplace()


In [ ]:
%sql
SELECT * FROM iceberg_catalog.iceberg_db.parking_iceberg LIMIT 10;


In [ ]:
%sql
SELECT * FROM iceberg_catalog.iceberg_db.parking_iceberg.history;


#### View Current Snapshot Metadata

%sql

SELECT * FROM iceberg_catalog.iceberg_db.parking_iceberg.metadata_log_entries;

#### Query Past Snapshot Before Append

In [ ]:
%sql
SELECT * FROM iceberg_catalog.iceberg_db.parking_iceberg.metadata_log_entries;

##### Append New Records (e.g., new violation records)

In [ ]:
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType, DateType
from datetime import datetime


In [ ]:
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType, DateType

schema = StructType([
    StructField("Summons_Number", LongType(), True),
    StructField("Plate_ID", StringType(), True),
    StructField("Registration_State", StringType(), True),
    StructField("Plate_Type", StringType(), True),
    StructField("Issue_Date", DateType(), True),
    StructField("Violation_Code", IntegerType(), True),
    StructField("Vehicle_Body_Type", StringType(), True),
    StructField("Vehicle_Make", StringType(), True),
    StructField("Issuing_Agency", StringType(), True),
    StructField("Vehicle_Expiration_Date", IntegerType(), True),
    StructField("Violation_Location", IntegerType(), True),
    StructField("Violation_Precinct", IntegerType(), True),
    StructField("Issuer_Precinct", IntegerType(), True),
    StructField("Issuer_Code", IntegerType(), True),
    StructField("Issuer_Command", StringType(), True),
    StructField("Issuer_Squad", StringType(), True),
    StructField("Time_First_Observed", StringType(), True),
    StructField("Violation_County", StringType(), True),
    StructField("Violation_In_Front_Of_Or_Opposite", StringType(), True),
    StructField("House_Number", StringType(), True),
    StructField("Street_Name", StringType(), True),
    StructField("Intersecting_Street", StringType(), True),
    StructField("Law_Section", IntegerType(), True),
    StructField("Sub_Division", StringType(), True),
    StructField("Violation_Legal_Code", StringType(), True),
    StructField("Days_Parking_In_Effect", StringType(), True),
    StructField("From_Hours_In_Effect", StringType(), True),
    StructField("To_Hours_In_Effect", StringType(), True),
    StructField("Vehicle_Color", StringType(), True),
    StructField("Unregistered_Vehicle", StringType(), True),
    StructField("Vehicle_Year", IntegerType(), True),
    StructField("Meter_Number", StringType(), True),
    StructField("Feet_From_Curb", IntegerType(), True),
    StructField("Violation_Post_Code", StringType(), True),
    StructField("Violation_Description", StringType(), True),
    StructField("Parking_Violation_Time", StringType(), True),
    StructField("Days_Parking_in_Full_Effect", StringType(), True)
])


In [ ]:
data = [
    (
        9999999999,
        'NEW1234',
        'New York',
        'Passenger',
        datetime.strptime('2025-05-28', '%Y-%m-%d').date(),
        40,
        'SDN',
        'TESLA',
        'P',
        20260101,
        120,
        120,
        120,
        123456,
        '0120',
        '0000',
        '0100P',
        'Queens',
        'F',
        '99-99',
        'FUTURE AVE',
        None,
        408,
        'F1',
        None,
        'BBBBBBB',
        'ALL',
        'ALL',
        'WHITE',
        '0',
        2025,
        '-',
        2,
        None,
        None,
        '1:30 PM',
        'No days'
    )
]


In [ ]:
# 'data' should be a list of tuples/rows matching the schema, e.g. [ (val1, val2, ...), ... ]
new_rows_df = spark.createDataFrame(data, schema=schema)


In [ ]:
%sql
SELECT COUNT(*) 
FROM iceberg_catalog.iceberg_db.parking_iceberg VERSION AS OF <old_snapshot_id>;


In [ ]:
new_rows_df.writeTo("iceberg_catalog.iceberg_db.parking_iceberg").append()


In [ ]:
%sql
SELECT * FROM iceberg_catalog.iceberg_db.parking_iceberg.metadata_log_entries;


In [ ]:
%sql
SELECT COUNT(*) FROM iceberg_catalog.iceberg_db.parking_iceberg VERSION AS OF <new-snapshot_id>;


In [ ]:
%sql
-- Time travel to past version
SELECT * FROM iceberg_catalog.iceberg_db.parking_iceberg
VERSION AS OF <old-snapshot-id>;


##### Query Using Time Travel

%sql
-- Time travel to past version
SELECT * FROM iceberg_catalog.iceberg_db.parking_iceberg
VERSION AS OF <old_snapshot_id>;


In [ ]:
%sql
SELECT COUNT(*) FROM iceberg_catalog.iceberg_db.parking_iceberg
VERSION AS OF <latest_snapshot_id>;


##### Delete Specific Records, Show Snapshot Diff

In [ ]:
%sql
-- Delete some rows
DELETE FROM iceberg_catalog.iceberg_db.parking_iceberg WHERE Registration_State = 'New York';


#### Then Compare Snapshots

%sql
SELECT COUNT(*) FROM iceberg_catalog.iceberg_db.parking_iceberg VERSION AS OF <old-snapshot-id>;


In [ ]:
%sql
SELECT COUNT(*) FROM iceberg_catalog.iceberg_db.parking_iceberg VERSION AS OF <new-snapshot-id>;


##### Metadata Changes

In [ ]:
%sql
-- Check metadata files associated with snapshot
SELECT * FROM iceberg_catalog.iceberg_db.parking_iceberg.history;


##### Perform a Rollback to Previous Version (Disaster Recovery)

In [ ]:
%sql
CREATE OR REPLACE TABLE iceberg_catalog.iceberg_db.parking_iceberg AS
SELECT *
FROM iceberg_catalog.iceberg_db.parking_iceberg VERSION AS OF <old-snapshot-id>;


In [ ]:
%sql
SELECT count(*) FROM iceberg_catalog.iceberg_db.parking_iceberg_restored;


In [ ]:
%sql
-- Check metadata files associated with snapshot
SELECT * FROM iceberg_catalog.iceberg_db.parking_iceberg.metadata_log_entries;


In [ ]:
%sql
-- Compare specific snapshots of an Iceberg table
SELECT *
FROM iceberg_catalog.iceberg_db.parking_iceberg.snapshots
WHERE snapshot_id IN (<old_snapshot_id>, <new_snapshot_id>)
ORDER BY committed_at;


In [ ]:
%sql
SELECT COUNT(*) FROM iceberg_catalog.iceberg_db.parking_iceberg;
